# Notebook 1: Dataset Setup and EDA

**Dataset:** CBIS-DDSM (Curated Breast Imaging Subset of DDSM)  
**Goal:** Load metadata CSVs, verify image paths, binarise labels, and produce a cleaned linked dataset for downstream modelling.

## 1. Environment Setup
Import project config and standard libraries. Config centralises all paths, label mappings, and constants.

In [ ]:
#Import config and standard libraries
import sys
sys.path.append(
    '/kaggle/input/datasets/mfjmrizvi/cbis-ddsm-project-config'
)
import config as cfg
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import json
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf

print(f"TensorFlow: {tf.__version__}")
print(f"GPUs: {tf.config.list_physical_devices('GPU')}")
print(f"\nConfig imported successfully")
print(f"  PNG_ROOT:    {cfg.PNG_ROOT}")
print(f"  PATCH_SIZE:  {cfg.PATCH_SIZE}")
print(f"  LABEL_MAP:   {cfg.LABEL_MAP}")
print(f"  RANDOM_SEED: {cfg.RANDOM_SEED}")

## 2. Path Verification
Confirm all dataset CSVs and the JPEG image root are accessible before loading.

In [ ]:
#Verify all paths exist

print("PATH VERIFICATION")
paths = {
    'MASS_TRAIN_CSV': cfg.MASS_TRAIN_CSV,
    'MASS_TEST_CSV':  cfg.MASS_TEST_CSV,
    'METADATA_CSV':   cfg.METADATA_CSV,
    'PNG_ROOT':       cfg.PNG_ROOT,
}

all_ok = True
for name, path in paths.items():
    exists = os.path.exists(path)
    print(f"  {'✓' if exists else '✗ MISSING'} {name}")
    if not exists:
        all_ok = False

print(f"\nAll paths OK: {all_ok}")


## 3. Load Raw Metadata
CBIS-DDSM provides separate CSVs for mass and calcification cases (train/test splits).  
`dicom_info.csv` is used later to resolve image UIDs to JPEG file paths.

In [ ]:
#Load all CSV files

mass_train_raw = pd.read_csv(cfg.MASS_TRAIN_CSV)
mass_test_raw  = pd.read_csv(cfg.MASS_TEST_CSV)
metadata_df    = pd.read_csv(cfg.METADATA_CSV)

print("=== SHAPES ===")
print(f"mass_train: {mass_train_raw.shape}")
print(f"mass_test:  {mass_test_raw.shape}")
print(f"metadata:   {metadata_df.shape}")

## 4. Schema Inspection
Mass and calcification CSVs share most columns but differ in lesion-specific fields  
(`mass shape` / `mass margins` vs. `calc type` / `calc distribution`).

In [ ]:
#Inspect column names

print("=== MASS TRAIN COLUMNS ===")
for i, col in enumerate(mass_train_raw.columns):
    print(f"  [{i:2d}] '{col}'")

## 5. Label Distribution
Three raw pathology classes: `MALIGNANT`, `BENIGN`, `BENIGN_WITHOUT_CALLBACK`.  
Both benign classes are collapsed to `0` in the binary task.

In [ ]:
#Pathology label inspection

print("=== MASS TRAIN PATHOLOGY ===")
print(mass_train_raw['pathology'].value_counts())
print("Unique:", mass_train_raw['pathology'].unique())

print("\n=== MASS TEST PATHOLOGY ===")
print(mass_test_raw['pathology'].value_counts())

## 6. Missing Value Audit
Identify nulls in lesion-descriptor columns. These are descriptive only and do not affect image loading or labels.

In [ ]:
#Missing value audit

print("=== MISSING VALUES — MASS TRAIN ===")
m_missing = mass_train_raw.isnull().sum()
print(m_missing[m_missing > 0])
print(f"Total: {m_missing.sum()}")

## 7. Label Binarisation
Map `MALIGNANT → 1` and both benign classes `→ 0` using the config's `binarise_pathology()` helper.

In [ ]:
#Binarise labels using config function

mass_train = cfg.binarise_pathology(mass_train_raw)
mass_test  = cfg.binarise_pathology(mass_test_raw)

print("=== BINARY LABEL DISTRIBUTION ===")
print("\nMass Train:")
print(mass_train['label'].value_counts())
imb = (mass_train['label'].value_counts()[0] /
       mass_train['label'].value_counts()[1])
print(f"Imbalance ratio: {imb:.2f}:1 (benign:malignant)")

print("\nMass Test:")
print(mass_test['label'].value_counts())

print("\nNull labels in mass_train:",
      mass_train['label'].isnull().sum())


## 8. Class Distribution Plot
Visualise label balance across all four splits. Near-balanced mass train (1.07:1) means no resampling is required.

In [ ]:
# Class distribution plot

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('CBIS-DDSM Pathology Distribution',
             fontsize=16, fontweight='bold')

datasets = [
    (mass_train, 'Mass Train', axes[0]),
    (mass_test,  'Mass Test',  axes[1]),
]

colors = ['#e74c3c', '#2ecc71', '#3498db']

for df, title, ax in datasets:
    counts = df['pathology'].value_counts()
    bars = ax.bar(counts.index, counts.values,
                  color=colors[:len(counts)])

    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Pathology')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=30)

    for bar, count in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 5,
                str(count),
                ha='center',
                va='bottom',
                fontweight='bold')

plt.tight_layout()
plt.savefig(f'{cfg.OUTPUT}/pathology_distribution.png',
            dpi=150, bbox_inches='tight')
plt.show()

print("Plot saved")

## 9. Build Image Path Maps
`dicom_info.csv` maps study UIDs to JPEG filenames. Separate maps are built for full mammograms and cropped ROI patches.

In [ ]:
full_path_map    = cfg.build_path_map_from_metadata(metadata_df, 'full mammogram images', cfg.PNG_ROOT, 'full')
cropped_path_map = cfg.build_path_map_from_metadata(metadata_df, 'ROI mask images', cfg.PNG_ROOT, 'crop')
roi_path_map      = cfg.build_path_map_from_metadata(metadata_df, 'ROI mask images', cfg.PNG_ROOT, 'mask')

print(f"ROI mask path map:        {len(roi_path_map)} entries")
print(f"Full mammogram path map:  {len(full_path_map)} entries")
print(f"Cropped image path map:   {len(cropped_path_map)} entries")

if full_path_map:
    uid = list(full_path_map.keys())[0]
    print(f"\nSample full path exists: {os.path.exists(full_path_map[uid])}")
if cropped_path_map:
    uid = list(cropped_path_map.keys())[0]
    print(f"Sample cropped path exists: {os.path.exists(cropped_path_map[uid])}")
if roi_path_map:
    uid = list(roi_path_map.keys())[0]
    print(f"Sample ROI mask path exists: {os.path.exists(roi_path_map[uid])}")

## 10. Link Image Paths to Metadata Rows
Resolve the `image file path` and `cropped image file path` columns in the mass CSVs to absolute JPEG paths on disk.

In [ ]:
#Link full image paths to mass CSV rows

full_paths_train = cfg.link_column_to_paths(
    mass_train, 'image file path', full_path_map
)
full_paths_test = cfg.link_column_to_paths(
    mass_test, 'image file path', full_path_map
)

mass_train['full_image_path'] = full_paths_train
mass_test['full_image_path']  = full_paths_test

train_linked = mass_train['full_image_path'].notna().sum()
test_linked  = mass_test['full_image_path'].notna().sum()

print(f"Full images linked (train): {train_linked}/{len(mass_train)}")
print(f"Full images linked (test):  {test_linked}/{len(mass_test)}")

In [ ]:
#Link cropped image paths to mass CSV rows

cropped_paths_train = cfg.link_column_to_paths(
    mass_train, 'cropped image file path', cropped_path_map
)
cropped_paths_test = cfg.link_column_to_paths(
    mass_test, 'cropped image file path', cropped_path_map
)

mass_train['cropped_image_path'] = cropped_paths_train
mass_test['cropped_image_path']  = cropped_paths_test

train_cropped = mass_train['cropped_image_path'].notna().sum()
test_cropped  = mass_test['cropped_image_path'].notna().sum()

print(f"Cropped images linked (train): {train_cropped}/{len(mass_train)}")
print(f"Cropped images linked (test):  {test_cropped}/{len(mass_test)}")

# Verify file existence
sample = mass_train[mass_train['cropped_image_path'].notna()].iloc[0]
print(f"\nSample cropped path: {sample['cropped_image_path']}")
print(f"File exists: {os.path.exists(sample['cropped_image_path'])}")

In [ ]:
#Link ROI mask paths to mass CSV rows

roi_paths_train = cfg.link_column_to_paths(
    mass_train, 'ROI mask file path', roi_path_map
)
roi_paths_test = cfg.link_column_to_paths(
    mass_test, 'ROI mask file path', roi_path_map
)

mass_train['roi_mask_path'] = roi_paths_train
mass_test['roi_mask_path']  = roi_paths_test

print(f"ROI masks linked (train): {mass_train['roi_mask_path'].notna().sum()}/{len(mass_train)}")
print(f"ROI masks linked (test):  {mass_test['roi_mask_path'].notna().sum()}/{len(mass_test)}")

sample = mass_train[mass_train['roi_mask_path'].notna()].iloc[0]
print(f"\nSample ROI mask path: {sample['roi_mask_path']}")
print(f"Same file as cropped image? "
      f"{sample['roi_mask_path'] == sample['cropped_image_path']}")

## 11. Drop Unlinked Rows & Finalise
Remove any rows where either image path is missing. All 1318 train and 378 test rows survive, confirming complete linkage.

In [ ]:
#Drop unlinked rows and verify final state

mass_train_clean = mass_train[
    mass_train['full_image_path'].notna() &
    mass_train['cropped_image_path'].notna()
].reset_index(drop=True)

mass_test_clean = mass_test[
    mass_test['full_image_path'].notna() &
    mass_test['cropped_image_path'].notna()
].reset_index(drop=True)

print(f"mass_train after cleaning: {len(mass_train_clean)}")
print(f"mass_test after cleaning:  {len(mass_test_clean)}")

print(f"\nFinal label distribution (train):")
print(mass_train_clean['label'].value_counts())

print(f"\nFinal columns:")
print(mass_train_clean.columns.tolist())

# ROI mask availability
train_roi_avail = mass_train_clean['roi_mask_path'].notna().sum()
test_roi_avail  = mass_test_clean['roi_mask_path'].notna().sum()
print(f"\nROI masks available (train): {train_roi_avail}/{len(mass_train_clean)}")
print(f"ROI masks available (test):  {test_roi_avail}/{len(mass_test_clean)}")

## 12. Patient-Level Audit
In MIL, each patient maps to one bag. Patients with multiple lesions
need a consistent bag-level label strategy

In [ ]:
# Patient-level abnormality audit
obs_per_patient = mass_train_clean.groupby('patient_id').size()
print(f"Max abnormalities per patient: {obs_per_patient.max()}")
print(f"Patients with multiple lesions: {(obs_per_patient > 1).sum()}")
print(f"Total unique patients: {mass_train_clean['patient_id'].nunique()}")

In [ ]:
#  5-fold patient-grouped, stratified split 
from sklearn.model_selection import StratifiedGroupKFold

N_FOLDS = 5
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=cfg.RANDOM_SEED)

fold_assignment = np.full(len(mass_train_clean), -1, dtype=int)
for fold_i, (_, val_idx) in enumerate(
        sgkf.split(mass_train_clean, mass_train_clean['label'],
                   groups=mass_train_clean['patient_id'])):
    fold_assignment[val_idx] = fold_i

mass_train_clean['fold'] = fold_assignment
assert (mass_train_clean['fold'] >= 0).all(), "Some rows not assigned to a fold"

print(f"Total rows: {len(mass_train_clean)}  |  Total patients: "
      f"{mass_train_clean['patient_id'].nunique()}")
print()
for f in range(N_FOLDS):
    fold_df       = mass_train_clean[mass_train_clean['fold'] == f]
    fold_patients = set(fold_df['patient_id'])
    other_patients = set(mass_train_clean.loc[mass_train_clean['fold'] != f, 'patient_id'])
    assert not (fold_patients & other_patients), f"LEAK: fold {f} shares patients with rest"
    print(f"Fold {f}: {len(fold_df):4d} rows | {len(fold_patients):3d} patients | "
          f"label dist {fold_df['label'].value_counts().to_dict()}")

test_patients = set(mass_test_clean['patient_id'])
train_patients_all = set(mass_train_clean['patient_id'])
assert not (train_patients_all & test_patients), "LEAK: train/test"
print(f"\nTest: {len(mass_test_clean)} rows, {len(test_patients)} patients")
print("✓ No patient overlap: any fold vs rest, or train vs test")

## 13. Save Outputs
Export the cleaned, path-linked CSVs and an EDA summary JSON for use in subsequent notebooks.

In [ ]:
# Save linked CSVs as notebook outputs
mass_train_clean.to_csv(f'{cfg.OUTPUT}/mass_train_linked.csv', index=False)
mass_test_clean.to_csv(f'{cfg.OUTPUT}/mass_test_linked.csv', index=False)

print("Saved:")
print(f"  {cfg.OUTPUT}/mass_train_linked.csv : {len(mass_train_clean)} rows, "
      f"'fold' column 0-{N_FOLDS-1}")
print(f"  {cfg.OUTPUT}/mass_test_linked.csv  : {len(mass_test_clean)} rows (held out)")

In [ ]:
#Save EDA summary JSON

summary = {
    'dataset':       'CBIS-DDSM (awsaf49 Kaggle)',
    'labels_source': 'Official TCIA download',
    'scope':         'Mass cases only (prototype phase)',
    'label_mapping': cfg.LABEL_MAP,
    'mass_train': {
        'total_rows':        len(mass_train_clean),
        'label_0_benign':    int(
            mass_train_clean['label'].value_counts()[0]),
        'label_1_malignant': int(
            mass_train_clean['label'].value_counts()[1]),
        'full_linked':       int(
            mass_train_clean['full_image_path'].notna().sum()),
        'cropped_linked':    int(
            mass_train_clean['cropped_image_path'].notna().sum()),
    },
    'mass_test': {
        'total_rows':        len(mass_test_clean),
        'label_0_benign':    int(
            mass_test_clean['label'].value_counts()[0]),
        'label_1_malignant': int(
            mass_test_clean['label'].value_counts()[1]),
    },
    'missing_values': {
        'mass_shape':   4,
        'mass_margins': 43,
        'impact':       'Low — descriptive columns only'
    }
}

with open(f'{cfg.OUTPUT}/eda_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("EDA summary saved")
print(json.dumps(summary, indent=2))